In [1]:
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import polars as pl
import sys

# Identify the project root (one level up from /research)
root = Path.cwd().parent
src_path = str(root / "src")

if src_path not in sys.path:
    sys.path.append(src_path)

# Now try the import—it should look for 'predictor' inside 'src'
from predictor.atmospheric.cleaners import AtmosphericCleaner
cleaner = AtmosphericCleaner()

# Setup Paths
RAW_DATA = Path("../data/raw")
LIS_PATH = RAW_DATA / "LIS_Sep2020//isslis_v3_fin_3-20260114_061035"
MODIS_FILE = RAW_DATA / "MODIS_Sep2020/202009MODIS.csv"
PAN_CO_FILE = RAW_DATA / "PAN_CO_Sep2020/cleaned/PAN_CO_0902_0930.csv"

### Data Cleaning

In [2]:
df_pan = pd.read_csv(PAN_CO_FILE)
print(df_pan.columns)
# Check if 'pressure' or 'altitude' exists to apply your 700-300hPa limit
if 'pressure' in df_pan.columns:
    df_pan = df_pan[(df_pan['pressure'] <= 700) & (df_pan['pressure'] >= 300)]

Index(['Date', 'Lng', 'Lat', 'CO', 'PAN'], dtype='object')


#### Lightning

In [3]:
lis_files = list(LIS_PATH.glob("*.nc"))
print(f"Found {len(lis_files)} files to process.")

Found 478 files to process.


In [4]:
all_dfs = []
for f in lis_files:
    try:
        df = cleaner.clean_lis_netcdf(f)
        if not df.is_empty():
            all_dfs.append(df)
    except Exception:
        continue

if all_dfs:
    # 1. Combine
    master_lightning = pl.concat(all_dfs)
    
    # 2. Conversion (Math on Floats)
    tai93_unix_offset = 725846400 
    master_lightning = master_lightning.with_columns(
        ((pl.col("tai_time") + tai93_unix_offset) * 1_000_000)
        .cast(pl.Int64)
        .cast(pl.Datetime("us"))
        .alias("datetime")
    )
    
    # 3. SORT is mandatory for group_by_dynamic
    daily_stats = (
        master_lightning
        .sort("datetime")  # <--- This fixes the InvalidOperationError
        .group_by_dynamic("datetime", every="1d")
        .agg([
            pl.len().alias("strike_count"),
            pl.col("lat").mean().alias("mean_lat"),
            pl.col("lat").std().alias("std_lat")
        ])
    ).with_columns([
        pl.col("strike_count").rolling_mean(window_size=3).alias("lightning_3d_smooth")
    ])
    
    print(f"✅ FINAL SUCCESS: {master_lightning.height} lightning events in memory.")
    print(daily_stats.head())
else:
    print("❌ Critical: Loop ran but list is empty.")

✅ FINAL SUCCESS: 3016099 lightning events in memory.
shape: (5, 5)
┌─────────────────────┬──────────────┬───────────┬───────────┬─────────────────────┐
│ datetime            ┆ strike_count ┆ mean_lat  ┆ std_lat   ┆ lightning_3d_smooth │
│ ---                 ┆ ---          ┆ ---       ┆ ---       ┆ ---                 │
│ datetime[μs]        ┆ u32          ┆ f32       ┆ f32       ┆ f64                 │
╞═════════════════════╪══════════════╪═══════════╪═══════════╪═════════════════════╡
│ 2020-08-30 00:00:00 ┆ 2398         ┆ 20.448257 ┆ 10.105644 ┆ null                │
│ 2020-08-31 00:00:00 ┆ 77458        ┆ 17.995668 ┆ 19.03433  ┆ null                │
│ 2020-09-01 00:00:00 ┆ 86260        ┆ 16.214483 ┆ 21.619715 ┆ 55372.0             │
│ 2020-09-02 00:00:00 ┆ 58679        ┆ 21.641605 ┆ 11.902352 ┆ 74132.333333        │
│ 2020-09-03 00:00:00 ┆ 41245        ┆ 15.641693 ┆ 19.481075 ┆ 62061.333333        │
└─────────────────────┴──────────────┴───────────┴───────────┴─────────────────────

In [5]:
# Save processed lightning data to data/processed/
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

out_path = PROCESSED_DIR / "lightning_sep2020_processed.parquet"
master_lightning.write_parquet(out_path)
print(f"Saved {master_lightning.height:,} events → {out_path}")

Saved 3,016,099 events → ../data/processed/lightning_sep2020_processed.parquet
